# DBSCAN: density-based clustering

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
A cluster is a **dense region** of points separated from others by sparse regions. DBSCAN does not need k, finds clusters of **any shape**, and labels lonely points as **noise**.

## Two parameters and three kinds of points
- `eps` ($\varepsilon$): the neighborhood radius.
- `min_samples`: the minimum number of points in the neighborhood (including the point itself).

| Kind | Definition |
|---|---|
| **Core** | Has at least `min_samples` points within eps |
| **Border** | Not core, but in the neighborhood of a core point |
| **Noise** | Neither core nor border |

## The algorithm
- Find an unlabeled core point; start a new cluster.
- Add every point reachable from it through a chain of core points (each step shorter than eps).
- Repeat; the remaining points are noise.

[[lab:dbscan]]

## A worked example
1-D points 1, 2, 3, 8, 9, 20 with $\varepsilon=1.5$ and min_samples = 2:

| Point | Neighbors within 1.5 | Kind |
|---|---|---|
| 1 | 1, 2 | Core |
| 2 | 1, 2, 3 | Core |
| 3 | 2, 3 | Core |
| 8 | 8, 9 | Core |
| 9 | 8, 9 | Core |
| 20 | 20 | **Noise** |

Two clusters: {1,2,3} and {8,9}; 20 is noise.



In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN

x = np.array([[1], [2], [3], [8], [9], [20]])
db = DBSCAN(eps=1.5, min_samples=2).fit(x)
print("labels (-1 = noise):", db.labels_)
print("core points:", x[db.core_sample_indices_].ravel())



> [eps too small: everything is noise. Right size: two moons. Too large: one cluster.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/dbscan_eps.png)

*eps too small: everything is noise. Right size: two moons. Too large: one cluster.*

## Choosing eps: the k-distance plot
For every point, compute the distance to its k-th neighbor (k = min_samples), sort and plot. Where the curve suddenly rises (the **knee**) is a good eps.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons
from sklearn.neighbors import NearestNeighbors

X, _ = make_moons(300, noise=0.07, random_state=0)
k = 5
dist, _ = NearestNeighbors(n_neighbors=k).fit(X).kneighbors(X)
d = np.sort(dist[:, -1])
plt.plot(d); plt.ylabel(f"distance to neighbor {k}"); plt.xlabel("points, sorted"); plt.title("k-distance plot: look for the knee"); plt.show()

for eps in [0.05, 0.1, 0.2, 0.5]:
    lab = DBSCAN(eps=eps, min_samples=k).fit_predict(X)
    print(f"eps={eps}: clusters={lab.max() + 1}, noise={np.sum(lab == -1)}")



| Pros | Cons |
|---|---|
| Any shape | Struggles with clusters of different densities |
| Detects noise | Sensitive to eps |
| No k needed | Weak in high dimensions |

> For varying densities, try `HDBSCAN` (in scikit-learn since 1.3); it needs no eps.

### Exercise

Try HDBSCAN on the same moons data.

In [ ]:
# Your answer here